# 90 Power BI export

Not a step in the analysis: this writes the results as tidy tables for a Power BI report, in a star schema (one table
per thing we describe, such as products, dates and forecast models, and fact tables that point to them). The table
definitions are in `powerbi/data_dictionary.md`, the relationships in `powerbi/star_schema.md`, the measures in
`powerbi/dax_measures.md` and the page layout in `powerbi/layout_spec.md`.

| table | one row per |
|---|---|
| pbi_dim_item | California product (all 3,049; `in_scope` marks the 726 we modelled) |
| pbi_dim_date | calendar day |
| pbi_dim_policy | forecast model and version |
| pbi_fact_sales_weekly | product and week |
| pbi_fact_breaks | product and forecast origin |
| pbi_fact_forecast_daily | product, day, model and version (scored origins) |
| pbi_fact_forecast_error | product, origin, model and version |
| pbi_fact_inventory_kpi | product, policy, target and lead time |
| pbi_fact_inventory_daily | product, day, version and target (7-day lead time) |
| pbi_fact_cost_sensitivity | cost scenario, policy and target |

In [1]:
import sys
sys.path.append('../src')
from config import BEST_NAME
BEST_SHORT = BEST_NAME.split(" (")[0]      # the best time-series model from 07h, stored as 'best'
import numpy as np
import pandas as pd
from break_detection import to_weekly

OUT = "../outputs/tables/"
fc = pd.read_parquet("../data/processed/backtest_forecasts.parquet")
err = pd.read_csv("../data/processed/backtest_errors.csv", parse_dates=['origin'])
breaks = pd.read_csv("../data/processed/breaks_by_origin.csv",
                     parse_dates=['origin', 'first_date', 'last_change_week', 'last_alarm_week', 'adaptive_start', 'placebo_start'])
cusum = pd.read_csv("../data/processed/cusum_breaks.csv", parse_dates=['change_week', 'alarm_week'])
agree = pd.read_csv("../data/processed/break_method_agreement.csv")
kpis = pd.read_csv("../data/processed/inventory_item_kpis.csv")
inv_daily = pd.read_parquet("../data/processed/inventory_daily.parquet")
sens = pd.read_csv("../outputs/tables/inventory_cost_sensitivity.csv")
calendar = pd.read_csv("../data/raw/calendar.csv", parse_dates=['date'])
ORIGINS = sorted(fc['origin'].unique())
EVAL_ORIGINS = ORIGINS[1:]
MODEL = kpis.loc[kpis['model'] != 'seasonal_naive', 'model'].iloc[0]
print("policy model:", MODEL)

policy model: best


In [2]:
# ---- dim_item
seg = pd.concat([pd.read_csv(f"../data/interim/{c}_segmentation.csv") for c in ['foods', 'hobbies', 'household']])
dim_item = seg[['item_id', 'abc_class', 'xyz_class', 'segment', 'mean_sales', 'cv', 'total_sales']].copy()
dim_item['category'] = dim_item['item_id'].str.split('_').str[0]
dim_item['dept_id'] = dim_item['item_id'].str.rsplit('_', n=1).str[0]
dim_item['in_scope'] = dim_item['item_id'].isin(fc['item_id'].unique())

last = breaks[breaks['origin'] == ORIGINS[-1]].set_index('item_id')
dim_item = dim_item.merge(cusum[['item_id', 'break_detected', 'change_week', 'direction', 'shift_in_sd']]
                          .rename(columns={'break_detected': 'cusum_first_break', 'change_week': 'first_break_week',
                                           'direction': 'first_break_direction', 'shift_in_sd': 'first_break_shift_sd'}),
                          on='item_id', how='left')
dim_item = dim_item.merge(agree[['item_id', 'pelt_n_changepoints']], on='item_id', how='left')
dim_item['break_at_last_origin'] = dim_item['item_id'].map(last['has_break'])
dim_item['n_breaks_at_last_origin'] = dim_item['item_id'].map(last['n_breaks'])
dim_item['last_break_week'] = dim_item['item_id'].map(last['last_change_week'])
dim_item['intermittent'] = dim_item['item_id'].map(err.groupby('item_id')['intermittent'].first())
dim_item['break_group'] = np.select([~dim_item['in_scope'], dim_item['break_at_last_origin'] == True],
                                    ['out of scope', 'break detected'], 'no break')
dim_item.to_csv(OUT + "pbi_dim_item.csv", index=False)

# ---- dim_date
dim_date = calendar[['date', 'wm_yr_wk', 'weekday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA']].copy()
dim_date['week_start'] = dim_date['date'] - pd.to_timedelta((dim_date['date'].dt.dayofweek - 5) % 7, unit='D')
dim_date['is_event'] = dim_date['event_name_1'].notna()
dim_date['is_eval_period'] = dim_date['date'].between(EVAL_ORIGINS[0] + pd.Timedelta(days=1), pd.Timestamp('2016-05-22'))
dim_date['forecast_origin'] = pd.cut(dim_date['date'], bins=list(ORIGINS) + [pd.Timestamp('2016-05-22')],
                                     labels=[o.date().isoformat() for o in ORIGINS], right=True).astype(str)
dim_date.to_csv(OUT + "pbi_dim_date.csv", index=False)

# ---- dim_policy
dim_policy = pd.DataFrame([(m, v) for m in ['seasonal_naive', 'ets', 'best', 'tsb'] for v in ['static', 'adaptive', 'placebo']],
                          columns=['model', 'variant'])
dim_policy['policy'] = dim_policy['model'] + '_' + dim_policy['variant']
dim_policy['used_in_inventory'] = dim_policy['policy'].isin(kpis['policy'].unique())
dim_policy.to_csv(OUT + "pbi_dim_policy.csv", index=False)
print(dim_item['break_group'].value_counts().to_dict())

{'out of scope': 2323, 'break detected': 488, 'no break': 238}


In [3]:
# ---- facts
long_df = pd.concat([pd.read_csv(f"../data/interim/{c}_candidates.csv", parse_dates=['date'])
                     for c in ['foods', 'hobbies', 'household']])
sales_weekly = pd.concat([to_weekly(g).rename('units').rename_axis('week_start').reset_index().assign(item_id=item)
                          for item, g in long_df.groupby('item_id')])
sales_weekly.to_csv(OUT + "pbi_fact_sales_weekly.csv", index=False)

fb = breaks[['item_id', 'origin', 'has_break', 'n_breaks', 'last_change_week', 'last_alarm_week', 'adaptive_window_days']].copy()
fb['is_eval_origin'] = fb['origin'].isin(EVAL_ORIGINS)
fb.to_csv(OUT + "pbi_fact_breaks.csv", index=False)

keep = fc['origin'].isin(EVAL_ORIGINS) & (fc['variant'] != 'placebo') & \
       ~((fc['model'] == 'seasonal_naive') & (fc['variant'] == 'adaptive'))
ffd = fc.loc[keep, ['item_id', 'date', 'origin', 'model', 'variant', 'actual', 'forecast', 'lo_90', 'hi_90']].copy()
ffd['policy'] = ffd['model'] + '_' + ffd['variant']
ffd['abs_error'] = (ffd['actual'] - ffd['forecast']).abs()
ffd.round(3).to_csv(OUT + "pbi_fact_forecast_daily.csv", index=False)

fe = err[['item_id', 'origin', 'model', 'variant', 'has_break', 'mase', 'rmsse', 'mae', 'mase_scale', 'dollar_weight']].copy()
fe['policy'] = fe['model'] + '_' + fe['variant']
fe['is_eval_origin'] = fe['origin'].isin(EVAL_ORIGINS)
fe.round(5).to_csv(OUT + "pbi_fact_forecast_error.csv", index=False)

kpis.round(4).to_csv(OUT + "pbi_fact_inventory_kpi.csv", index=False)
inv_daily.assign(policy=MODEL + '_' + inv_daily['variant']).round(3).to_csv(OUT + "pbi_fact_inventory_daily.csv", index=False)
sens.round(2).to_csv(OUT + "pbi_fact_cost_sensitivity.csv", index=False)

import os
for f in sorted(os.listdir(OUT)):
    if f.startswith('pbi_'):
        n = sum(1 for _ in open(OUT + f)) - 1
        print(f"{f:35s} {n:>9,d} rows  {os.path.getsize(OUT + f) / 1e6:6.1f} MB")

C:\Users\Anushka\AppData\Local\Temp\ipykernel_19404\4180151070.py:17: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ffd.round(3).to_csv(OUT + "pbi_fact_forecast_daily.csv", index=False)


C:\Users\Anushka\AppData\Local\Temp\ipykernel_19404\4180151070.py:22: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  fe.round(5).to_csv(OUT + "pbi_fact_forecast_error.csv", index=False)


C:\Users\Anushka\AppData\Local\Temp\ipykernel_19404\4180151070.py:25: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  inv_daily.assign(policy=MODEL + '_' + inv_daily['variant']).round(3).to_csv(OUT + "pbi_fact_inventory_daily.csv", index=False)


pbi_dim_date.csv                        1,969 rows     0.1 MB
pbi_dim_item.csv                        3,049 rows     0.4 MB
pbi_dim_policy.csv                         12 rows     0.0 MB
pbi_fact_breaks.csv                     5,082 rows     0.3 MB
pbi_fact_cost_sensitivity.csv             324 rows     0.0 MB
pbi_fact_forecast_daily.csv           618,576 rows    56.5 MB
pbi_fact_forecast_error.csv            46,284 rows     4.9 MB


pbi_fact_inventory_daily.csv          731,808 rows    59.5 MB
pbi_fact_inventory_kpi.csv             19,602 rows     3.3 MB
pbi_fact_sales_weekly.csv             183,968 rows     5.7 MB


## What we found

Nothing new: this only reshapes results from the earlier notebooks. The printed row counts are the check that every table was written.

## What comes next

`91_dashboard_export` writes the smaller files the dashboard reads.

In [4]:
# files this notebook wrote
from pathlib import Path
ROOT = Path('..').resolve()
written = [ROOT / p for p in ['outputs/tables/pbi_dim_item.csv', 'outputs/tables/pbi_dim_date.csv', 'outputs/tables/pbi_dim_policy.csv', 'outputs/tables/pbi_fact_sales_weekly.csv', 'outputs/tables/pbi_fact_breaks.csv', 'outputs/tables/pbi_fact_forecast_daily.csv', 'outputs/tables/pbi_fact_forecast_error.csv', 'outputs/tables/pbi_fact_inventory_kpi.csv', 'outputs/tables/pbi_fact_inventory_daily.csv', 'outputs/tables/pbi_fact_cost_sensitivity.csv']]
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\pbi_dim_item.csv 369 KB
outputs\tables\pbi_dim_date.csv 126 KB
outputs\tables\pbi_dim_policy.csv 0 KB
outputs\tables\pbi_fact_sales_weekly.csv 5,717 KB
outputs\tables\pbi_fact_breaks.csv 294 KB
outputs\tables\pbi_fact_forecast_daily.csv 56,511 KB
outputs\tables\pbi_fact_forecast_error.csv 4,948 KB
outputs\tables\pbi_fact_inventory_kpi.csv 3,251 KB
outputs\tables\pbi_fact_inventory_daily.csv 59,522 KB
outputs\tables\pbi_fact_cost_sensitivity.csv 14 KB
